# Runner sin hardware: replay del grafo LangGraph sobre CSV

Notebook pedagógico para la defensa de tesis: reproduce el mismo bucle de control que usa
el dron en vuelo real, pero reemplazando los dos únicos nodos de hardware con datos del CSV
de auditoría.

**Arquitectura de replay:**
```
CSV row[i] ──► CsvReplayClient.capture()   ──► capture_node      ┐
                                                                 ├─ grafo idéntico
CSV row[i] ──► _csv_perception_node()      ──► obstacle_field    ┘  a producción
                                            + señales V3/V4/G1 del CSV
                                                                         ▼
               CsvReplayClient.execute_velocity()  ◄── motor_node   (sin vuelo)
```

**Lo que es idéntico a producción:**
- `policy_router` con sus umbrales TTC/FOV/TRAJ_STALL/stuck_invisible reales del `.env`
- `keep_going`, `evasive` (con C1 stall lateral), `deliberative_node`, `girar_90_node` (con D1), `fsm_node`
- `WaypointTracker` con los waypoints reales de la misión
- El estado fluye ciclo a ciclo exactamente igual — no hay reconstrucción
- `motor_node` con corrección de altitud durante FRENAR (`_hover_alt_anchor`)

**Lo que difiere:**
- `perception_node` lee `field_*` del CSV (FlowTTC necesita imágenes reales)
- Las señales V3/V3b/V3c (`imu_contact_event`, `blind_wall_event`, `stuck_invisible`) se
  restauran desde columnas `ctrl_*` del CSV — el sensor IMU no está disponible en replay
- Las tasas de stall de trayectoria (`_traj_frente/izq/der_stall_rate`) se reconstruyen
  ciclo a ciclo desde el FlightTrajectory del replay; la frontal se sobreescribe con
  `ctrl_traj_stall_rate` del CSV para máxima fidelidad con producción
- V4 profundidad monocular (DepthEstimator) se sustituye por `ctrl_depth_m/cycles` del CSV
- Invocaciones SLM lanzan el VLM con los mismos prompts pero respuestas **nuevas**
- Los contadores de estado inicial (`evasion_stuck_cycles`, `maneuver_cycles_left`) parten en 0

**Cómo usar:**
1. Ajustar `CSV_PATH` y `MISSION_PATH` en §1
2. Ajustar `WINDOW_START` / `WINDOW_END` para la ventana de ciclos
3. Kernel → Run All Cells

<img src="../informe/2026-0904_grafo_control_dronelm_horizontal.png"/>

## §0 — Imports y configuración del runner sin hardware

In [ ]:
import sys, json, math, io, contextlib, os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# ── src/ al path ───────────────────────────────────────────────────────────────
SRC_DIR = str(Path('src').resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ── Importar piezas del grafo de producción ────────────────────────────────────
from src.agents.graph import _build_nodes, DroneState, degraded_router, policy_router
import src.agents.graph as _graph_mod
from src.agents.deliberative import _parse_decision, make_deliberative_node
from src.agents.deliberation_service import DeliberationResult
from src.agents.spatial_history import FlightTrajectory
from src.perception.obstacle_field import ObstacleField, Cell, SECTORS, BANDS
from src.navigation import WaypointTracker
from langgraph.graph import END, StateGraph

print(f'sys.path[0] = {sys.path[0]}')
print('Importaciones OK')

# ── CsvReplayClient ────────────────────────────────────────────────────────────
class CsvReplayClient:
    """Drop-in mock de AirSimClient: capture() y execute_velocity() sin hardware.

    Nota: imu_linear_acceleration no se reconstruye desde el CSV (solo se
    loguea el nivel resumido ctrl_imu_jitter, no ax/ay). El capture_node real
    calculará imu_contact_event=False; _csv_perception_node lo sobreescribirá
    desde ctrl_imu_contact del CSV, que es la señal registrada en producción.
    """

    def __init__(self):
        self._row: dict = {}
        self.last_command: dict = {}
        self._connected = True

    def connect(self): pass
    def takeoff(self): pass
    def land(self): pass

    def capture(self):
        row = self._row
        dummy_image = np.zeros((3, 3, 3), dtype=np.uint8)
        telemetry = {
            'source': 'airsim',
            'position': {'x': float(row.get('pos_x', 0.)), 'y': float(row.get('pos_y', 0.)), 'z': float(row.get('pos_z', 0.))},
            'velocity': {'vx': float(row.get('vel_x', 0.)), 'vy': float(row.get('vel_y', 0.)), 'vz': float(row.get('vel_z', 0.))},
            'orientation': {
                'yaw':   math.radians(float(row.get('yaw_deg',   0.))),
                'pitch': math.radians(float(row.get('pitch_deg', 0.))),
                'roll':  math.radians(float(row.get('roll_deg',  0.))),
            },
            'collision': {
                'has_collided': str(row.get('has_collided', 'False')).lower() == 'true',
                'object_name':  str(row.get('collision_object', '')),
            },
            'timestamp': float(row.get('t', 0.)),
            # IMU: no disponible en replay — capture_node calcula imu_contact_event=False;
            # _csv_perception_node sobreescribe con ctrl_imu_contact del CSV.
            'imu_linear_acceleration': {'ax': 0.0, 'ay': 0.0, 'az': 0.0},
        }
        return dummy_image, telemetry

    def execute_velocity(self, vx=0., vy=0., vz=0., yaw_rate=0., target_yaw=None):
        self.last_command = {'vx': vx, 'vy': vy, 'vz': vz, 'yaw_rate': yaw_rate, 'target_yaw': target_yaw}

    def get_telemetry(self):
        _, telem = self.capture()
        return telem


# ── ReplayDeliberationService ──────────────────────────────────────────────────
class ReplayDeliberationService:
    """Reemplaza DeliberationService para replay determinístico sin VLM.

    Protocolo:
    - request()                 → registra el pedido (sin llamar al modelo)
    - preload_for_resolution()  → carga la respuesta grabada ANTES del ciclo
                                  donde debe resolverse en producción
    - poll()                    → devuelve el resultado SOLO si fue precargado;
                                  mientras no lo sea, devuelve None (= pendiente)

    El criterio de resolución viene del caller (bucle de replay), no del servicio:
    se detecta comparando slm_delib_id entre filas del CSV.
    """

    def __init__(self):
        self._next_id   = 0
        self._pending_id: Optional[int] = None
        self._pending_at: Optional[float] = None
        self._resolution_raw: Optional[str] = None

    def preload_for_resolution(self, raw_response: str) -> None:
        self._resolution_raw = raw_response

    def clear_pending(self) -> None:
        self._pending_id     = None
        self._resolution_raw = None
        self._pending_at     = None

    def request(self, payload) -> int:
        self._next_id += 1
        self._pending_id  = self._next_id
        self._pending_at  = __import__('time').time()
        return self._next_id

    def poll(self):
        import time as _t
        age_ms = ((_t.time() - self._pending_at) * 1000.
                  if self._pending_at else 0.)
        if self._pending_id is not None and self._resolution_raw is not None:
            parsed = _parse_decision(self._resolution_raw)
            result = DeliberationResult(
                request_id    = self._pending_id,
                completed_at  = _t.time(),
                parsed_decision = parsed,
                raw_response  = self._resolution_raw,
                latency_ms    = 0.,
                error         = None,
            )
            self._resolution_raw = None
            return result, age_ms, True   # has_pending=True: deliberative_node limpia el id
        return None, age_ms, self._pending_id is not None

    def stop(self): pass
    def is_watchdog_expired(self): return False


def _build_field_from_row(row: dict) -> ObstacleField:
    """Construye ObstacleField desde columnas field_* del CSV (3 sectores × 3 bandas)."""
    cells = {}
    for s in SECTORS:
        occ     = float(row.get(f'field_{s}_occ', 0.) or 0.)
        ttc_raw = row.get(f'field_{s}_ttc_s')
        ttc     = float(ttc_raw) if ttc_raw not in (None, '', 'nan') else float('inf')
        conf    = float(row.get(f'field_{s}_conf', 0.) or 0.)
        for b in BANDS:
            cells[(s, b)] = Cell(sector=s, band=b, occupancy=occ, ttc_s=ttc, confidence=conf)
    return ObstacleField(cells=cells, source='flow')


def _csv_perception_node(state: DroneState) -> DroneState:
    """Reemplaza perception_node: ObstacleField + señales V3/V4/G1 del CSV.

    En producción, perception_node calcula estas señales en tiempo real:
      V3  — imu_contact_event desde aceleración IMU (ax/ay no disponibles en replay)
      V3b — blind_wall_event desde divergencia cmd vs velocidad real
      V3c — _stopped_cycles desde velocidad real ≈ 0 sostenida
      V4  — _depth_proximity_m desde DepthEstimator (modelo monocular, no ejecuta en replay)
      G1  — aviso en scene_summary cuando stall frontal con campo óptico libre

    En replay, todas estas señales se restauran desde columnas ctrl_* del CSV,
    que es exactamente lo que registró FlightLogger en la corrida de producción.
    Las tasas de stall de trayectoria izq/der se acumulan en tiempo real desde
    FlightTrajectory (capture_node real); la frontal se sobreescribe con el CSV.
    """
    row = replay_client._row

    # ObstacleField desde columnas field_*
    field = _build_field_from_row(row)
    state['obstacle_field'] = field
    state['estimated_ttc']  = field.min_ttc()
    state['scene_summary']  = field.summary_text()

    # V3 (IMU contact): captura_node calculó False (sin ax/ay reales) → restaurar del CSV
    state['imu_contact_event'] = str(row.get('ctrl_imu_contact', 'False')).lower() == 'true'
    state['imu_jitter_level']  = str(row.get('ctrl_imu_jitter', 'normal') or 'normal')

    # V3b (pared invisible por divergencia cmd/real):
    state['blind_wall_event'] = str(row.get('ctrl_blind_wall', 'False')).lower() == 'true'
    bw_raw = row.get('ctrl_blind_wall', 'False')  # usamos como proxy de _blind_wall_cycles
    state['_blind_wall_cycles'] = 1 if str(bw_raw).lower() == 'true' else 0

    # V3c (drone parado sin razón):
    state['_stopped_cycles'] = int(float(row.get('ctrl_stopped_cycles', 0) or 0))

    # Señal unificada (OR de V3/V3b/V3c):
    state['stuck_invisible'] = str(row.get('ctrl_stuck_invisible', 'False')).lower() == 'true'

    # V4 (profundidad monocular — DepthEstimator no corre en replay):
    d_m = row.get('ctrl_depth_m')
    state['_depth_proximity_m']  = float(d_m) if d_m not in (None, '', 'nan') else None
    state['_depth_below_cycles'] = int(float(row.get('ctrl_depth_cycles', 0) or 0))

    # TRAJ_STALL (C1): tasa de stall frontal del log (sobreescribe la acumulada en replay)
    state['_traj_frente_stall_rate'] = float(row.get('ctrl_traj_stall_rate', 0.) or 0.)
    state['_traj_frente_attempts']   = int(float(row.get('ctrl_traj_attempts', 0) or 0))

    # G1: aviso de muro invisible en scene_summary
    _g1_stall = state['_traj_frente_stall_rate']
    _g1_att   = state['_traj_frente_attempts']
    _G1_STALL_MIN = float(os.getenv('G1_STALL_MIN', '0.20'))
    _G1_ATT_MIN   = int(os.getenv('G1_ATT_MIN', '3'))
    _G1_OCC_MAX   = float(os.getenv('G1_OCC_MAX', '0.15'))
    if (_g1_stall >= _G1_STALL_MIN and _g1_att >= _G1_ATT_MIN
            and field.blocked_fraction() < _G1_OCC_MAX):
        state['scene_summary'] = (
            state.get('scene_summary', '')
            + f'\nAVISO [G1]: stall frontal {_g1_stall:.0%}'
              f' ({_g1_att} intentos) con campo optico despejado'
              f' — posible obstaculo invisible (muro liso, baja textura).'
              f' MANTENER_RUMBO agravara el bloqueo.'
              f' Priorizar GIRAR_90 o evasion lateral amplia.'
        )
    return state


# ── Construir grafo con mocks ──────────────────────────────────────────────────
replay_client        = CsvReplayClient()
replay_delib_service = ReplayDeliberationService()
replay_trajectory    = FlightTrajectory()   # acumula stalls ciclo a ciclo en replay

_nodes = _build_nodes(replay_client)
_nodes['perception']   = _csv_perception_node
# make_deliberative_node(service, trajectory): trajectory=replay_trajectory permite
# que el deliberative_node en replay use el historial de trayectoria acumulado,
# igual que en producción (aunque empieza vacío, lo que es lo previsto).
_nodes['deliberative'] = make_deliberative_node(replay_delib_service, replay_trajectory)

_wf = StateGraph(DroneState)
for _name, _fn in [
    ('capture',        _nodes['capture']),
    ('degraded_hover', _nodes['degraded_hover']),
    ('perception',     _nodes['perception']),
    ('keep_going',     _nodes['keep_going']),
    ('evasive',        _nodes['evasive']),
    ('deliberative',   _nodes['deliberative']),
    ('girar_90',       _nodes['girar_90']),
    ('fsm',            _nodes['fsm']),
    ('motor',          _nodes['motor']),
]:
    _wf.add_node(_name, _fn)

_wf.set_entry_point('capture')
_wf.add_conditional_edges('capture', degraded_router,
    {'degraded_hover': 'degraded_hover', 'perception': 'perception'})
_wf.add_conditional_edges('perception', policy_router,
    {'keep_going': 'keep_going', 'evasive': 'evasive',
     'deliberative': 'deliberative', 'girar_90': 'girar_90', 'fsm': 'fsm'})
for _e in ('degraded_hover', 'keep_going', 'evasive', 'deliberative', 'girar_90', 'fsm'):
    _wf.add_edge(_e, 'motor')
_wf.add_edge('motor', END)

app = _wf.compile()
print('Grafo compilado')
print('  capture     → real (FlightTrajectory acumulada, IMU=zeros → imu_contact override en perception)')
print('  perception  → CSV (field_* + ctrl_* para V3/V3b/V3c/V4/G1/TRAJ_STALL)')
print('  deliberative → ReplayDeliberationService (sin VLM, respuestas del log)')
print('  motor       → sin AirSim (CsvReplayClient.execute_velocity)')

## §1 — Cargar corrida y misión

In [2]:
# ── CAMBIAR AQUÍ ───────────────────────────────────────────────────────────────
CSV_PATH     = '../airsim-runs/produccion/tier1/townsim_clear/slm/deep_vlm/seed_1.csv'
MISSION_PATH = '../airsim-plan/missions/flightplans/townsim_clear.json'
WINDOW_START = 0    # primer índice CSV a reproducir (0-based)
WINDOW_END   = 60   # último índice inclusive  (None = hasta el final del CSV)
# ───────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
ARM      = df['arm'].iloc[0]
SCENARIO = df['scenario'].iloc[0]
SEED     = int(df['seed'].iloc[0])

if WINDOW_END is None:
    WINDOW_END = len(df) - 1

print(f'CSV: {len(df)} ciclos  |  arm={ARM}  scenario={SCENARIO}  seed={SEED}')
print(f'Ventana de replay: índices {WINDOW_START}–{WINDOW_END}  ({WINDOW_END - WINDOW_START + 1} ciclos)')

# Cargar misión y crear WaypointTracker
with open(MISSION_PATH, encoding='utf-8') as f:
    mission = json.load(f)
waypoints = mission.get('waypoints', [])
waypoint_tracker = WaypointTracker(waypoints)
print(f'Misión cargada: {len(waypoints)} waypoints → {[list(wp.values())[:3] for wp in waypoints[:3]]} ...')

# Apuntar AGENT_ARM al brazo del CSV (policy_router lo lee del módulo)
_graph_mod.AGENT_ARM = ARM
print(f'AGENT_ARM → {ARM}')

CSV: 1324 ciclos  |  arm=slm  scenario=townsim_clear  seed=1
Ventana de replay: índices 0–60  (61 ciclos)
Misión cargada: 6 waypoints → [[0.0, 0.4, -10.0], [0.4, -71.6, -30.0], [-145.2, -70.8, -30.0]] ...
AGENT_ARM → slm


## §2 — Replay de la ventana de ciclos

Reproduce los ciclos `WINDOW_START`–`WINDOW_END` usando el grafo real.
El estado fluye de ciclo en ciclo igual que en producción — sin reconstrucciones artificiales.

In [ ]:
import time as _time

# Estado inicial — igual que main.py antes del primer graph.invoke()
drone_state: DroneState = {
    'waypoints':            waypoints,
    'current_wp_index':     int(df.iloc[WINDOW_START].get('wp_index', 0)),
    'target_waypoint':      None,
    'waypoint_guidance':    {},
    'mission_completed':    False,
    'rgb_image':            None,
    'telemetry':            {},
    'frame_history':        [],
    'frame_history_ts':     [],
    'estimated_ttc':        float('inf'),
    'next_action':          '',
    'flight_status':        'vuelo',
    'deliberations':        [],
    'active_maneuver':      None,
    'maneuver_cycles_left': 0,
    'maneuver_command':     None,
    'evasion_stuck_cycles': 0,
    'slm_request_id':       None,
    'route':                '',
    # V3/V3b/V3c — señales de obstáculo invisible
    'imu_contact_event':    False,
    'imu_jitter_level':     'normal',
    '_imu_contact_cycles':  0,
    'blind_wall_event':     False,
    '_blind_wall_cycles':   0,
    '_stopped_cycles':      0,
    'stuck_invisible':      False,
    # V4 — profundidad monocular
    '_depth_proximity_m':   None,
    '_depth_below_cycles':  0,
    '_depth_obstacle_type': None,
    # TRAJ_STALL (C1) — historia de stalls de trayectoria
    '_traj_frente_stall_rate': 0.0,
    '_traj_frente_attempts':   0,
    '_traj_izq_stall_rate':    0.0,
    '_traj_izq_attempts':      0,
    '_traj_der_stall_rate':    0.0,
    '_traj_der_attempts':      0,
    # motor_node — corrección de altitud durante FRENAR
    '_hover_alt_anchor':    None,
}

# Detección de columnas ctrl_* (FlightLogger ≥ 2026-0909)
_HAS_CTRL_COLS = 'ctrl_slm_pending' in df.columns
# Detección de columnas V3/V4 (FlightLogger ≥ Zona-1 simplificación)
_HAS_V3_COLS   = 'ctrl_stuck_invisible' in df.columns
_HAS_TRAJ_COLS = 'ctrl_traj_stall_rate' in df.columns

# Inicializar servicio de deliberación limpio
replay_delib_service.clear_pending()

# Si la ventana empieza en el medio del CSV, recuperar el estado de control
# del ciclo anterior (el que "entra" al ciclo WINDOW_START)
if WINDOW_START > 0:
    _pre = df.iloc[WINDOW_START - 1]
    if _HAS_CTRL_COLS:
        drone_state['evasion_stuck_cycles']  = int(_pre.get('ctrl_stuck_cycles', 0) or 0)
        man = str(_pre.get('ctrl_active_maneuver', '') or '')
        drone_state['active_maneuver']       = man if man else None
        drone_state['maneuver_cycles_left']  = int(_pre.get('ctrl_maneuver_cycles_left', 0) or 0)
        if str(_pre.get('ctrl_slm_pending', 'False')).lower() == 'true':
            drone_state['slm_request_id'] = -1  # centinela: hay pedido pendiente

results = []

for idx in range(WINDOW_START, WINDOW_END + 1):
    row      = df.iloc[idx]
    prev_row = df.iloc[idx - 1] if idx > WINDOW_START else None

    # ── 1. Restaurar estado de control desde el final del ciclo anterior ──────
    if prev_row is not None:
        if _HAS_CTRL_COLS:
            drone_state['evasion_stuck_cycles']  = int(prev_row.get('ctrl_stuck_cycles', 0) or 0)
            man = str(prev_row.get('ctrl_active_maneuver', '') or '')
            drone_state['active_maneuver']       = man if man else None
            drone_state['maneuver_cycles_left']  = int(prev_row.get('ctrl_maneuver_cycles_left', 0) or 0)
            prev_pending = str(prev_row.get('ctrl_slm_pending', 'False')).lower() == 'true'
            if not prev_pending:
                # El ciclo anterior terminó sin pedido pendiente → limpiar
                drone_state['slm_request_id'] = None
                replay_delib_service.clear_pending()
        else:
            # Sin ctrl_*: nueva deliberación empieza cuando la ruta anterior NO era deliberative
            if str(prev_row['route']) != 'deliberative':
                drone_state['slm_request_id'] = None
                replay_delib_service.clear_pending()

    # ── 2. Precargar respuesta del SLM si este ciclo la resuelve en producción ─
    # Criterio universal (funciona con y sin ctrl_*):
    # cuando slm_delib_id CAMBIA de una fila a la siguiente, la deliberación
    # fue resuelta en este ciclo (se añadió una entrada nueva a deliberations[]).
    curr_delib_id = str(row.get('slm_delib_id',  '') or '')
    prev_delib_id = str((prev_row.get('slm_delib_id', '') or '') if prev_row is not None else '')
    if curr_delib_id and curr_delib_id != prev_delib_id:
        replay_delib_service.preload_for_resolution(str(row.get('slm_raw_response', '')))

    # ── 3. Apuntar el mock al CSV ─────────────────────────────────────────────
    replay_client._row = row.to_dict()

    # ── 4. Sincronizar WaypointTracker ───────────────────────────────────────
    pos_now = {'x': float(row['pos_x']), 'y': float(row['pos_y']), 'z': float(row['pos_z'])}
    yaw_now = math.radians(float(row.get('yaw_deg', 0.)))
    if hasattr(waypoint_tracker, 'current_index'):
        try:
            waypoint_tracker.current_index = int(row.get('wp_index', 0))
        except (AttributeError, TypeError):
            pass
    target_wp = waypoint_tracker.update(pos_now)
    guidance  = waypoint_tracker.compute_guidance(pos_now, yaw_now)
    drone_state['target_waypoint']   = target_wp
    drone_state['waypoint_guidance'] = guidance
    drone_state['current_wp_index']  = waypoint_tracker.current_index

    # ── 5. Invocar el grafo ───────────────────────────────────────────────────
    try:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            drone_state = app.invoke(drone_state)
        rep_route  = drone_state.get('route', '?')
        rep_action = drone_state.get('next_action', '?')
        error = None
    except Exception as exc:
        rep_route  = 'ERROR'
        rep_action = '?'
        error      = str(exc)[:80]

    csv_route  = str(row['route'])
    csv_action = str(row['action'])
    match = rep_route == csv_route

    results.append({
        'idx':        idx,
        'cycle':      int(row['cycle']),
        't_s':        float(row['t']),
        'route_rep':  rep_route,
        'route_csv':  csv_route,
        'action_rep': rep_action,
        'action_csv': csv_action,
        'match':      match,
        'centro_blk': str(row['field_centro_blocked']) if 'field_centro_blocked' in row.index else str(row.get('field_centro_occ', '?')),
        'centro_ttc': str(row.get('field_centro_ttc_s', ''))[:7],
        'delib_id':   curr_delib_id,
        'slm_res':    bool(curr_delib_id and curr_delib_id != prev_delib_id),
        'stuck_cyc':  int(drone_state.get('evasion_stuck_cycles', 0)),
        'stuck_inv':  bool(drone_state.get('stuck_invisible')),
        'blind_wall': bool(drone_state.get('blind_wall_event')),
        'traj_stall': float(drone_state.get('_traj_frente_stall_rate', 0.)),
        'depth_m':    drone_state.get('_depth_proximity_m'),
        'error':      error,
    })

df_res = pd.DataFrame(results)
n_match = int(df_res['match'].sum())
n_total = len(df_res)
pct     = 100 * n_match / n_total if n_total else 0

modes = []
if _HAS_CTRL_COLS:  modes.append('ctrl_* (control)')
if _HAS_V3_COLS:    modes.append('V3/V3b/V3c')
if _HAS_TRAJ_COLS:  modes.append('TRAJ_STALL')
mode_str = ' + '.join(modes) if modes else '⚠  CSV anterior (sin ctrl_*)'
print(f'[{mode_str}] — SLM: respuestas del log, sin llamadas al VLM')
print(f'Replay: {n_match}/{n_total} ciclos coinciden ({pct:.1f}%)\n')
display(df_res[['idx', 'cycle', 't_s', 'route_rep', 'route_csv', 'match',
                'centro_blk', 'centro_ttc', 'delib_id', 'slm_res',
                'stuck_cyc', 'stuck_inv', 'blind_wall', 'traj_stall']].to_string(index=False))

## §3 — Inspección profunda de un ciclo

Selecciona un ciclo de la tabla anterior y muestra el estado de entrada, el razonamiento
paso a paso del `policy_router`, y el resultado comparado con producción.

In [ ]:
# ── CAMBIAR AQUÍ ───────────────────────────────────────────────────────────────
INSPECT_IDX = WINDOW_START   # índice CSV del ciclo a inspeccionar
# ───────────────────────────────────────────────────────────────────────────────

from src.perception.obstacle_field import empty_field, has_open_corridor
from src.navigation.waypoint_tracker import effective_stall_threshold, hard_stall_threshold

TTC_EV  = float(os.getenv('TTC_EVASION_THRESHOLD', '3.2'))
TTC_SA  = float(os.getenv('TTC_SAFE_THRESHOLD',    '4.6'))
FOV_BK  = float(os.getenv('FOV_BLOCKED_THRESHOLD', '0.6'))
OPT_MIN = float(os.getenv('OPTICAL_MIN_ALT_M',     '4.5'))
SLM_MIN = float(os.getenv('SLM_MIN_ALT_M',         '8.0'))
_TRAJ_STALL_TRIGGER  = float(os.getenv('TRAJ_STALL_TRIGGER_RATE',   '0.70'))
_TRAJ_ATT_TRIGGER    = int(os.getenv('TRAJ_STALL_TRIGGER_MIN_ATT', '10'))

row_i  = df.iloc[INSPECT_IDX]
print(f'Ciclo {int(row_i["cycle"])}  (t={float(row_i["t"]):.1f}s  arm={row_i["arm"]})')

# Reconstruir ObstacleField con Cell objects (misma función que usa _csv_perception_node)
field_i = _build_field_from_row(row_i.to_dict())

# Guía de waypoint para este ciclo
pos_i  = {'x': float(row_i['pos_x']), 'y': float(row_i['pos_y']), 'z': float(row_i['pos_z'])}
yaw_i  = math.radians(float(row_i.get('yaw_deg', 0.)))
guid_i = waypoint_tracker.compute_guidance(pos_i, yaw_i)

# Resultado del replay para este ciclo
row_res    = df_res[df_res['idx'] == INSPECT_IDX]
rep_route  = row_res['route_rep'].iloc[0]  if not row_res.empty else '?'
rep_action = row_res['action_rep'].iloc[0] if not row_res.empty else '?'

print()
print('═' * 66)
print('  ESTADO DE ENTRADA')
print('═' * 66)
print(f'  route anterior        = {row_i.get("route", "?")}')
print(f'  evasion_stuck_cycles  = {row_res["stuck_cyc"].iloc[0] if not row_res.empty else "?"}')
print(f'  active_maneuver       = {drone_state.get("active_maneuver")}  '
      f'(cycles_left={drone_state.get("maneuver_cycles_left", 0)})')
print()
print('  ObstacleField:')
for s in SECTORS:
    occ  = field_i.sector_occupancy(s)
    ttc  = field_i.sector_ttc(s)
    blk  = field_i.is_blocked(s)
    conf = field_i.sector_confidence(s)
    ttc_str = f'{ttc:.2f}' if ttc != float('inf') else 'inf'
    print(f'    {s:<12}  occ={occ:.2f}  ttc={ttc_str}  conf={conf:.2f}  blk={blk}')
print(f'  WP guidance:  bearing_err={guid_i.get("bearing_err_deg",0.):.1f}°  '
      f'dist_xy={guid_i.get("dist_xy",0.):.1f}m')

print()
print('  Señales V3/V3b/V3c/V4 (del CSV):')
print(f'    imu_contact_event   = {str(row_i.get("ctrl_imu_contact", "False")).lower() == "true"}  '
      f'(jitter={row_i.get("ctrl_imu_jitter","?")})')
print(f'    blind_wall_event    = {str(row_i.get("ctrl_blind_wall", "False")).lower() == "true"}')
print(f'    _stopped_cycles     = {int(float(row_i.get("ctrl_stopped_cycles", 0) or 0))}  '
      f'(umbral={os.getenv("STOPPED_CYCLES_THRESHOLD","15")})')
print(f'    stuck_invisible     = {str(row_i.get("ctrl_stuck_invisible", "False")).lower() == "true"}')
d_m = row_i.get('ctrl_depth_m')
d_str = f'{float(d_m):.2f}m' if d_m not in (None, '', 'nan') else 'None'
print(f'    _depth_proximity_m  = {d_str}  '
      f'(_depth_below_cycles={int(float(row_i.get("ctrl_depth_cycles",0) or 0))})')

print()
print('═' * 66)
print('  POLICY_ROUTER — trazado de condiciones')
print('═' * 66)
print(f'  AGENT_ARM = {_graph_mod.AGENT_ARM}')

if _graph_mod.AGENT_ARM == 'reactive':
    print('  → keep_going  (brazo reactive: sin deliberación)')
elif _graph_mod.AGENT_ARM == 'fsm':
    print('  → fsm  (brazo fsm: máquina de estados determinista)')
else:
    imu_ct    = str(row_i.get('ctrl_imu_contact', 'False')).lower() == 'true'
    bw_ev     = str(row_i.get('ctrl_blind_wall',  'False')).lower() == 'true'
    stuck_inv = str(row_i.get('ctrl_stuck_invisible', 'False')).lower() == 'true'

    ttc_min       = field_i.min_ttc()
    c_blocked     = field_i.is_blocked('centro')
    c_ttc         = field_i.sector_ttc('centro')
    fov_frac      = field_i.blocked_fraction()
    open_corr     = has_open_corridor(field_i, guid_i)
    c_imminent    = c_ttc != float('inf') and c_ttc <= TTC_EV

    stuck         = int(drone_state.get('evasion_stuck_cycles', 0))
    active_man    = drone_state.get('active_maneuver')
    cyc_left      = int(drone_state.get('maneuver_cycles_left', 0))

    traj_stall    = float(row_i.get('ctrl_traj_stall_rate', 0.) or 0.)
    traj_att      = int(float(row_i.get('ctrl_traj_attempts', 0) or 0))

    pos_z         = abs(float(pos_i.get('z', 0.)))
    below_optical = pos_z < OPT_MIN
    below_slm     = pos_z < SLM_MIN

    print(f'  slm_request_id          = {drone_state.get("slm_request_id")}')
    print(f'  active_maneuver         = {active_man}  cycles_left={cyc_left}')
    print(f'  evasion_stuck_cycles    = {stuck}  '
          f'(eff_stall={effective_stall_threshold()}  hard={hard_stall_threshold()})')
    print()
    print('  Orden de evaluación del router:')
    print(f'  [1] imu_contact_event={imu_ct} OR blind_wall_event={bw_ev}'
          f'  → {"⇒ evasive ✓" if imu_ct or bw_ev else "continúa..."}')
    print(f'  [2] slm_request_id={drone_state.get("slm_request_id")} is not None'
          f'  → {"⇒ deliberative ✓" if drone_state.get("slm_request_id") is not None else "continúa..."}')
    print(f'  [3] stuck_invisible={stuck_inv}'
          f'  → {"⇒ evasive ✓" if stuck_inv else "continúa..."}')
    print(f'  [4] below_optical_floor (|z|={pos_z:.1f}m < {OPT_MIN}m)={below_optical}'
          f'  → {"⇒ keep_going ✓" if below_optical else "continúa..."}')

    # [5] active_maneuver persistencia
    am_active = bool(active_man and cyc_left > 0)
    print(f'  [5] active_maneuver={active_man} cycles_left={cyc_left}'
          f'  → {"⇒ evasive ✓" if am_active else "continúa..."}')

    # [6] escape de deadlock
    if stuck >= effective_stall_threshold():
        hard_exit = stuck >= hard_stall_threshold() or not open_corr
        print(f'  [6] stuck={stuck}>={effective_stall_threshold()}: hard={hard_exit} open_corr={open_corr}'
              f'  → {"⇒ deliberative ✓" if hard_exit else "continúa (corr. abierto)..."}')
    else:
        print(f'  [6] stuck={stuck} < eff_stall={effective_stall_threshold()} → continúa...')

    # [7] TRAJ_STALL
    traj_triggered = traj_stall >= _TRAJ_STALL_TRIGGER and traj_att >= _TRAJ_ATT_TRIGGER
    print(f'  [7] TRAJ_STALL stall={traj_stall:.0%} att={traj_att}'
          f' (>={_TRAJ_STALL_TRIGGER:.0%}/{_TRAJ_ATT_TRIGGER}?)'
          f'  → {"⇒ deliberative ✓" if traj_triggered else "continúa..."}')

    # [8] TTC / occupancy
    ttc_str   = f'{ttc_min:.2f}' if ttc_min != float('inf') else 'inf'
    c_ttc_str = f'{c_ttc:.2f}'   if c_ttc   != float('inf') else 'inf'
    print(f'  [8] field.min_ttc()={ttc_str}  centro_ttc={c_ttc_str}'
          f'  centro_blocked={c_blocked}  fov_frac={fov_frac:.2f}')
    print(f'       below_slm_floor (|z|={pos_z:.1f}m < {SLM_MIN}m)={below_slm}')
    if c_imminent or (c_blocked and c_ttc <= TTC_SA):
        if fov_frac > FOV_BK:
            dest = 'evasive' if below_slm else 'girar_90'
        else:
            dest = 'evasive' if below_slm else 'deliberative'
        print(f'       centro_imminent={c_imminent} OR (c_blocked AND ttc<={TTC_SA})  → ⇒ {dest} ✓')
    elif c_blocked or ttc_min <= TTC_SA:
        print(f'       c_blocked OR ttc<={TTC_SA}  → ⇒ evasive ✓')
    else:
        print(f'       sin amenaza  → ⇒ keep_going ✓')

print()
print('═' * 66)
print('  RESULTADO')
print('═' * 66)
match_i = not row_res.empty and bool(row_res['match'].iloc[0])
print(f'  route reproducida   = {rep_route}')
print(f'  route en producción = {row_i["route"]}')
print(f'  action reproducida  = {rep_action}')
print(f'  action en producción= {row_i["action"]}')
print(f'  coincidencia        = {"✅" if match_i else "⚠️  divergencia"}')

## §4 — Análisis de divergencias

Ciclos donde la route reproducida difiere de la registrada en producción, con causa probable.

In [ ]:
df_div = df_res[~df_res['match']].copy()
print(f'Divergencias: {len(df_div)}/{len(df_res)} ciclos\n')

if df_div.empty:
    print('✅ Replay perfecto — sin divergencias en la ventana seleccionada.')
else:
    print(f'  {"idx":>5}  {"cycle":>6}  {"t_s":>6}  {"route_rep":<15}  {"route_csv":<15}  causa probable')
    print('  ' + '─' * 92)
    for _, rd in df_div.iterrows():
        if rd['error']:
            causa = f'EXCEPCIÓN: {rd["error"]}'
        elif rd['blind_wall']:
            causa = 'V3b: blind_wall_event activo → evasive en replay pero no en CSV (o viceversa)'
        elif rd['stuck_inv']:
            causa = 'V3c: stuck_invisible activo → evasive (drone parado sin razón)'
        elif rd['traj_stall'] >= float(os.getenv('TRAJ_STALL_TRIGGER_RATE', '0.70')):
            causa = 'TRAJ_STALL: stall frontal persistente → deliberative vía historial de trayectoria'
        elif rd['slm_res']:
            causa = 'SLM: VLM dio respuesta distinta en replay (nueva invocación)'
        elif rd['route_csv'] in ('evasive', 'girar_90') and rd['route_rep'] == 'keep_going':
            causa = 'Estado interno: evasion_stuck_cycles / maneuver_cycles_left ≠ producción'
        elif rd['route_csv'] == 'deliberative' and rd['route_rep'] != 'deliberative':
            causa = 'TTC/FOV en umbral límite — sensible a variación de frame'
        else:
            causa = 'Estado propagado desde ciclo anterior divergente'
        print(f'  {int(rd["idx"]):>5}  {int(rd["cycle"]):>6}  {float(rd["t_s"]):>6.1f}'
              f'  {rd["route_rep"]:<15}  {rd["route_csv"]:<15}  {causa}')

# Visualización match / mismatch por tiempo
if len(df_res) > 0:
    fig, ax = plt.subplots(figsize=(14, 2))
    colors = ['#4CAF50' if m else '#F44336' for m in df_res['match']]
    ax.bar(df_res['t_s'], [1] * len(df_res), color=colors, width=0.3, linewidth=0)
    ax.set_yticks([])
    ax.set_xlabel('Tiempo (s)')
    ax.set_title(f'Match por ciclo — {n_match}/{n_total} ({pct:.1f}%)   '
                 f'verde = coincide   rojo = divergencia')
    handles = [mpatches.Patch(color='#4CAF50', label='match'),
               mpatches.Patch(color='#F44336', label='divergencia')]
    ax.legend(handles=handles, fontsize=8)
    fig.tight_layout()
    plt.show()

## §5 — Diagrama del grafo LangGraph

Arquitectura del `StateGraph` que corre en producción y en el replay (los nodos marcados
con ⚡ y 🔇 son los únicos que difieren).

In [ ]:
diagram = """
graph TD
    capture["📷 capture_node\\ntelemetría del CSV\\n+ FlightTrajectory (stall rates)\\n+ IMU jitter (ax=ay=0→override)"]
    capture --> deg_r{degraded?}
    deg_r -->|sí| dh["degraded_hover\\n(flotar)"] --> motor
    deg_r -->|no| perc["⚡ perception_node\\nObstacleField del CSV\\n+ V3/V3b/V3c/V4/G1 del CSV"]
    perc --> pr{"policy_router\\narm + prioridades:\\n[1] imu_contact|blind_wall\\n[2] slm_request_id\\n[3] stuck_invisible\\n[4] below_optical_floor\\n[5] active_maneuver\\n[6] deadlock escape\\n[7] TRAJ_STALL\\n[8] TTC/FOV"}
    pr -->|reactive| kg["keep_going\\nguiar a waypoint"]
    pr -->|slm libre| delib["deliberative_node\\nconsultar VLM"]
    pr -->|maniobra| ev["evasive_node\\nmaniobra\\n(C1: stall lateral)"]
    pr -->|deadlock/TRAJ| g90["girar_90_node\\nescaneo 90°\\n(D1: historia stalls)"]
    pr -->|fsm| fsm["fsm_node\\nmáquina de estados"]
    kg    --> motor
    delib --> motor
    ev    --> motor
    g90   --> motor
    fsm   --> motor
    motor["🔇 motor_node\\nexecute_velocity() sin vuelo\\n+ corrección altitud FRENAR"]

    style capture fill:#E3F2FD,stroke:#1565C0
    style perc    fill:#FFF9C4,stroke:#F9A825
    style motor   fill:#E8F5E9,stroke:#2E7D32
    style pr      fill:#FCE4EC,stroke:#C62828
"""

display(Markdown(f'```mermaid{diagram}```'))

print()
print('Nodos modificados respecto a producción:')
print('  ⚡ perception_node  → lee field_* del CSV; señales V3/V3b/V3c/V4/G1 de ctrl_* del CSV')
print('  📷 capture_node     → CsvReplayClient.capture() devuelve telemetría del CSV (IMU=zeros)')
print('  🔇 motor_node       → CsvReplayClient.execute_velocity() sin AirSim')
print()
print('Nodos 100% idénticos a producción:')
print('  policy_router  [1-8 prioridades incluyendo TRAJ_STALL, V3/V3c/stuck_invisible]')
print('  keep_going   evasive_node (C1: stall lateral)   deliberative_node')
print('  girar_90_node (D1: historia stalls)   fsm_node')
print()
print('Señales nuevas en el grafo actual vs versión anterior del notebook:')
print('  V3  imu_contact_event  — colisión por vibración IMU (restore desde ctrl_imu_contact)')
print('  V3b blind_wall_event   — pared invisible por divergencia cmd/real (ctrl_blind_wall)')
print('  V3c _stopped_cycles    — drone parado sin razón (ctrl_stopped_cycles)')
print('  V4  _depth_proximity_m — profundidad monocular (ctrl_depth_m; sin DepthEstimator)')
print('  G1  scene_summary      — aviso muro invisible cuando stall frontal + campo libre')
print('  C1  stall lateral      — evasive_node penaliza lado con alta tasa de stall')
print('  D1  girar_90 con hist  — gira al lado contrario si el preferido tiene stall alto')
print('  TRAJ_STALL             — policy_router escalona a deliberative antes de deadlock duro')